# 15. Hybrid GAT + LightGBM (task 5.7, GĐ F)

GAT nhẹ (cấu hình task 5.3) cho embedding 64 chiều mỗi node; ghép với 18 đặc trưng đã chuẩn hóa thành 82 cột; LightGBM phân loại. Cấu hình LightGBM chọn trên **val** (seed 42) trong cùng không gian 3 cấu hình với baseline LightGBM.

- `model=hybrid_gat_lgbm`: embedding train **out-of-fold** (5 nhóm khối thời gian liên tiếp; mỗi nhóm lấy embedding từ GAT huấn luyện trên 4 nhóm còn lại). Val dùng GAT huấn luyện trên toàn bộ train.
- `model=hybrid_gat_lgbm_nooof`: embedding train lấy trực tiếp từ GAT huấn luyện trên toàn bộ train (để so sánh).
- 3 seed (42, 43, 44) × 2 bộ (NF-ToN-IoT-v3, NF-BoT-IoT-v3), chỉ `split=val`. **Không đọc test.**
- Cuối notebook: `compare_val_<bộ>.csv` (mean ± std F1-macro, FPR, recall tấn công cho mọi mô hình; đánh dấu hybrid có vượt mô hình tốt nhất hơn 2 std hay không), đọc kết quả val của baseline và GNN từ output của notebook 05, 07, 13, 14.

Gắn: output của 04, 11 (đồ thị), 05, 13 (baseline), 07, 14 (GNN). Bật Internet. Chỉ bật GPU khi Phúc đã xem giờ GPU còn lại và đồng ý. Mô hình lưu ở `/kaggle/working/models_hybrid/<bộ>_s<seed>/` cho GĐ G, H.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    """RAM đỉnh (GB) của tiến trình notebook và mọi tiến trình con đã kết thúc (ru_maxrss tính bằng KB trên Linux)."""
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()                # ghi sau mỗi bước, lỗi giữa chừng vẫn còn run_info.json
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"], "| CPU:", INFO["cpu_count"])

## 1. pytest toàn bộ (dừng nếu có test lỗi, gồm `tests/test_hybrid.py`)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Hybrid: mỗi bộ, cả out-of-fold và không out-of-fold, 3 seed

In [ ]:
DATASETS = ["nf_ton_iot_v3", "nf_bot_iot_v3"]
MODELS = f"{WORK}/models_hybrid"
for ds in DATASETS:
    data_dir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    res, search = f"{WORK}/results_hybrid_{ds}.csv", f"{WORK}/hybrid_search_{ds}.json"
    for p in (res, search):
        assert not os.path.exists(p), f"{p} đã có từ trước, run_hybrid sẽ nối thêm dòng trùng"
    print("=====", ds, "| dữ liệu:", data_dir)
    for oof in (1, 0):             # oof=1 trước: GAT toàn train được lưu, oof=0 dùng lại đúng GAT đó
        with step(f"hybrid_{ds}_oof{oof}"):
            run([sys.executable, "-I", "scripts/run_hybrid.py", data_dir, ds, res, search, f"oof={oof}",
                 f"models={MODELS}"], cwd=REPO)

## 3. So sánh với baseline và GNN trên val (đọc output của notebook 05, 07, 13, 14)

In [ ]:
def find_one(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"không thấy {name} trong /kaggle/input"
    import hashlib
    if len({hashlib.md5(open(h, "rb").read()).hexdigest() for h in hits}) > 1:
        raise AssertionError(f"{name}: nhiều file khác nội dung {hits}")    # cùng nội dung (mount trùng) thì dùng bản đầu
    return hits[0]

with step("compare_val"):
    for ds in DATASETS:
        srcs = [find_one(f"results_baselines_{ds}.csv"), find_one(f"results_gnn_{ds}.csv"), f"{WORK}/results_hybrid_{ds}.csv"]
        print("nguồn:", srcs)
        run([sys.executable, "-I", "scripts/compare_val.py", ds, f"{WORK}/compare_val_{ds}.csv", *srcs], cwd=REPO)

## 4. Kiểm tra tiêu chí 5.7 và run_info.json

Chỉ đếm dòng và đối chiếu; số liệu báo cáo lấy từ `compare_val_<bộ>.csv`.

In [ ]:
import pandas as pd

crit, summary = {}, {}
for ds in DATASETS:
    res = pd.read_csv(f"{WORK}/results_hybrid_{ds}.csv")
    cnt = res.groupby(["model", "seed"]).size()
    crit[ds] = {
        "du_2_mo_hinh_x_3_seed_moi_cap_1_dong": bool(len(res) == 6 and (cnt == 1).all() and
            set(res["model"]) == {"hybrid_gat_lgbm", "hybrid_gat_lgbm_nooof"} and set(res["seed"]) == {42, 43, 44}),
        "chi_split_val": set(res["split"]) == {"val"},
        "co_n_params_size_do_tre": bool(res[["n_params", "size_kb", "infer_ms_per_window"]].gt(0).all().all()),
        "size_kb_max": float(res["size_kb"].max()),
    }
    search = json.load(open(f"{WORK}/hybrid_search_{ds}.json", encoding="utf-8"))
    crit[ds]["so_lan_thu_le_20"] = all(len(search[m]["seeds"]["42"]["search"]) <= 20 for m in search)
    crit[ds]["co_oof_k5"] = len(search["hybrid_gat_lgbm"]["seeds"]["42"]["folds"]) == 5
    comp = pd.read_csv(f"{WORK}/compare_val_{ds}.csv")
    h = comp[comp["is_hybrid"]]
    keys = ("f1_macro_mean", "f1_macro_std", "fpr_mean", "fpr_std", "recall_attack_mean",
            "f1_macro_best_model", "f1_macro_diff_vs_best", "f1_macro_beats_2std",
            "fpr_best_model", "fpr_diff_vs_best", "fpr_beats_2std")
    summary[ds] = {r["model"]: {k: (bool(r[k]) if k.endswith("beats_2std") else r[k]) for k in keys} for _, r in h.iterrows()}
    summary[ds]["diag_train_vs_val_f1"] = {m: {s: v["diag"] for s, v in search[m]["seeds"].items()} for m in search}
    print(ds, json.dumps(summary[ds], ensure_ascii=False, indent=1, default=str))
INFO["criteria"] = crit
INFO["summary"] = summary
INFO["models_hybrid_dirs"] = sorted(os.listdir(MODELS))
save_info()
print("\nrun_info.json:")
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())
run(["du", "-sh", MODELS])